# Transform Point to Rear Axle Frame

Computes the vehicle base frame from the rear axle endpoints and ground points, then expresses a target point in that frame.

| Axis | Direction |
|------|-----------|
| Origin | rear axle center |
| **y** | left along rear axle, projected onto ground plane |
| **z** | up — normal to ground plane (PCA) |
| **x** | forward — right-hand: x = y × z |

In [ ]:
# ── INPUTS — edit these values ────────────────────────────────────────────────

# Point to express in the base_link frame (sensor / world coordinates)
point_to_transform = [1.5, 0.0, 1.7]

# Left and right ends of the rear axle (same coordinate frame as above)
axle_left  = [0.0,  1.0, 0.4]
axle_right = [0.0, -1.0, 0.4]

# Ground points sampled around the wheels — each entry is [x, y, z]
ground_points = [
    [0.0, -1.0, 0.0],
    [0.0,  1.0, 0.0],
    [3.0, -1.0, 0.0],
    [3.0,  1.0, 0.0]
]

# ── COMPUTATION ───────────────────────────────────────────────────────────────

import numpy as np

p = np.array(point_to_transform, dtype=float)
L = np.array(axle_left,  dtype=float)
R = np.array(axle_right, dtype=float)
G = np.array(ground_points, dtype=float)

# Origin: midpoint of rear axle
origin = (L + R) / 2.0

# z: normal to ground plane — smallest singular vector of centred ground points
G_centred = G - G.mean(axis=0)
_, _, Vt = np.linalg.svd(G_centred, full_matrices=False)
z_axis = Vt[-1]

# Flip z if the axle center is on the negative side (z must point up toward axle)
if np.dot(z_axis, origin - G.mean(axis=0)) < 0:
    z_axis = -z_axis

# y: left along rear axle, with z component removed so y lies in the ground plane
y_axis = L - R
y_axis -= np.dot(y_axis, z_axis) * z_axis
y_axis /= np.linalg.norm(y_axis)

# x: forward — right-handed system (y × z)
x_axis = np.cross(y_axis, z_axis)
x_axis /= np.linalg.norm(x_axis)

# Rotation matrix: columns are x, y, z basis vectors
R_mat = np.column_stack([x_axis, y_axis, z_axis])

# Express point in the rear-axle frame
p_base_link = R_mat.T @ (p - origin)

# ── OUTPUT ────────────────────────────────────────────────────────────────────

np.set_printoptions(precision=4, suppress=True)

print("Rear-axle coordinate frame")
print(f"  Origin   : {origin}")
print(f"  x (fwd)  : {x_axis}")
print(f"  y (left) : {y_axis}")
print(f"  z (up)   : {z_axis}")
print()
print(f"Input point  (input frame) : {p}")
print(f"Output point (base_link)   : {p_base_link}")